# Conditional Autoregressive: Local ZINC Generation

Use nearby stored molecules as components to generate from a sampled molecule's interpretation graph.

In [ ]:
from pathlib import Path
import runpy

roots = (Path.cwd(), *Path.cwd().parents)
candidates = [
    root / relative
    for root in roots
    for relative in (
        "notebooks/_bootstrap.py",
        "repos/abstractgraph-generative/notebooks/_bootstrap.py",
    )
]
bootstrap_path = next((path for path in candidates if path.is_file()), None)
if bootstrap_path is None:
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")
runpy.run_path(str(bootstrap_path))

In [ ]:
from nsppk import NSPPK

from abstractgraph.operators import add, compose, cycle, intersection_edges, name, tree
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator

In [ ]:
graphs, _ = ZINCLoader(on_error="skip").load(
    "zinc_250k",
    limit=1500,
    min_node_count=20,
    max_node_count=30,
)
print(f"Loaded {len(graphs)} molecules")

In [ ]:
decomposition = compose(
    intersection_edges(),
    add(compose(name("cycle"), cycle()), compose(name("tree"), tree())),
)
generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition,
    nbits=14,
    n_jobs=1,
)
generator.store(
    graphs,
    neighbor_vectorizer=NSPPK(radius=1, distance=4, connector=1, nbits=14),
)

In [ ]:
samples = generator.sample(n_samples=3, n_neighbors=30, random_state=0)
print(f"Generated {len(samples)} molecules from {len(generator.last_sampled_indices_)} seeds")
if samples:
    sources = [generator.stored_graphs_[idx] for idx in generator.last_sampled_indices_]
    draw_molecules(sources + samples, n_graphs_per_line=4)
else:
    print("No molecules generated; try a larger neighborhood or training set.")